# Soil resistance
 

In [ ]:
# | default_exp soil_resistance

In [ ]:
# | hide
from fastcore import *
from nbdev.showdoc import *

In [ ]:
# | export
import numpy as np
from plant_hydraulics.parameter_classes import PhysCon, Leaf, RootVar, Soil, Flux

```
Input: Soil layers (θ, ψ, K, b, Δz), root parameters, LAI
                    │
                    ▼
    ┌────────────────────────────────────────┐
    │  For each soil layer j = 1 to 11:      │
    │                                        │
    │  1. K_soil = Ksat · s^(2b+3)           │
    │     (soil hydraulic conductivity)      │
    │                                        │
    │  2. RLD = root biomass / (ρ · π·r²·Δz) │
    │     (root length density)              │
    │                                        │
    │  3. r_dist = 1/√(π·RLD)                │
    │     (half-distance between roots)      │
    │                                        │
    │  4. R_soil = ln(r_dist/r_root)         │
    │            / (2π·RLD·Δz·K)             │
    │     (soil-to-root radial resistance)   │
    │                                        │
    │  5. R_root = ρ_resist / (ρ_bio · Δz)   │
    │     (root tissue resistance)           │
    │                                        │
    │  6. R_layer = R_soil + R_root  (series)│
    │                                        │
    │  7. Sum conductances: Σ 1/R_layer      │
    │     (parallel across layers)           │
    │                                        │
    │  8. E_max,j = (ψ_soil,j − ψ_min)/R_j   │
    │     (max transpiration from layer)     │
    └────────────────────────────────────────┘
                    │
                    ▼
    ┌────────────────────────────────────────┐
    │  R_total = LAI / Σ(1/R_layer)          │
    │  (convert ground → leaf area)          │
    └────────────────────────────────────────┘
                    │
                    ▼
    ┌────────────────────────────────────────┐
    │  ψ_soil = Σ(ψ_j · E_max,j) / Σ E_max   │
    │  (transpiration-weighted potential)    │
    └────────────────────────────────────────┘
                    │
                    ▼
    ┌────────────────────────────────────────┐
    │  et_loss_j = E_max,j / Σ E_max         │
    │  (fractional uptake per layer)         │
    └────────────────────────────────────────┘
                    │
                    ▼
          Return rsoil, psi_soil, et_loss
```

In [ ]:
# | export
def soil_resistance(
    physcon: PhysCon, leaf: Leaf, rootvar: RootVar, soil: Soil, flux: Flux
) -> Flux:
    """
    Calculate soil hydraulic resistance, weighted soil water potential,
    and water uptake from each soil layer.

    For each soil layer, computes the belowground resistance as the sum
    of a soil-to-root radial resistance (based on soil hydraulic
    conductivity and root geometry) and a root-to-stem axial resistance
    (based on root tissue resistivity). Layers are treated as parallel
    pathways, so their conductances (1/resistance) are summed to obtain
    the total belowground conductance. The weighted soil water potential
    is computed by weighting each layer's matric potential by its
    maximum transpiration capacity.

    The soil-to-root radial resistance for each layer is:

        R_soil = ln(r_dist / r_root) / (2π · RLD · Δz · K_soil)

    where r_dist is the mean half-distance between roots, RLD is root
    length density, Δz is layer thickness, and K_soil is the
    unsaturated hydraulic conductivity. The root-to-stem axial
    resistance is:

        R_root = ρ_root / (ρ_biomass · Δz)

    where ρ_root is the root tissue hydraulic resistivity and ρ_biomass
    is the root biomass density. The total belowground resistance per
    unit ground area is converted to per unit leaf area by multiplying
    by LAI.

    The weighted soil water potential uses the maximum transpiration
    rate from each layer (ψ_soil,j - ψ_min) / R_j as weights, so that
    wetter, better-connected layers contribute more to the effective
    soil water supply.

    __Parameters:__

    - PhysCon : Physical constants:
        - grav : Gravitational acceleration (m/s2).
        - denh2o : Density of liquid water (kg/m3).
        - mmh2o : Molecular mass of water (kg/mol).

    - Leaf : Leaf parameters:
        - minl_wp : Minimum leaf water potential (MPa).

    - RootVar : Fine root parameters:
        - biomass : Fine root biomass (g biomass/m2).
        - radius : Fine root radius (m).
        - density : Fine root density (g biomass/m3 root).
        - resist : Hydraulic resistivity of root tissue (MPa · s · g/mmol H2O)

    - Soil : Soil profile variables:
        - nlevsoi : Number of soil layers.
        - h2osoi_vol : Volumetric water content for each layer (m3/m3).
        - psi : Matric potential for each layer (mm).
        - watsat : Volumetric water content at saturation (porosity)for each
                   layer (m3/m3).
        - hksat : Hydraulic conductivity at saturation for each layer (mm H2O/s).
        - bsw : Clapp and Hornberger "b" parameter for each layer (-).
        - rootfr : Fraction of roots in each layer (-).
        - dz : Thickness of each layer (m).

    - Flux : Flux variables:
        - lai : Canopy leaf area index (m2/m2).

    __Returns:__

    - Flux : Updated flux object with the following attributes:
        - rsoil : Soil hydraulic resistance (MPa · s · m2/mmol H2O).
        - psi_soil : Weighted soil water potential (MPa).
        - et_loss : Fraction of total transpiration from each soil layer (-).

    Parameters
    ----------
    physcon : PhysCon
        Physical constants:
        - grav : float
            Gravitational acceleration (m/s2).
        - denh2o : float
            Density of liquid water (kg/m3).
        - mmh2o : float
            Molecular mass of water (kg/mol).
    leaf : Leaf
        Leaf parameters:
        - minl_wp : float
            Minimum leaf water potential (MPa).
    rootvar : RootVar
        Fine root parameters:
        - biomass : float
            Fine root biomass (g biomass/m2).
        - radius : float
            Fine root radius (m).
        - density : float
            Fine root density (g biomass/m3 root).
        - resist : float
            Hydraulic resistivity of root tissue
            (MPa · s · g/mmol H2O).
    soil : Soil
        Soil profile variables:
        - nlevsoi : int
            Number of soil layers.
        - h2osoi_vol : list of float
            Volumetric water content for each layer (m3/m3).
        - psi : list of float
            Matric potential for each layer (mm).
        - watsat : list of float
            Volumetric water content at saturation (porosity)
            for each layer (m3/m3).
        - hksat : list of float
            Hydraulic conductivity at saturation for each layer
            (mm H2O/s).
        - bsw : list of float
            Clapp and Hornberger "b" parameter for each layer (-).
        - rootfr : list of float
            Fraction of roots in each layer (-).
        - dz : list of float
            Thickness of each layer (m).
    flux : Flux
        Flux variables:
        - lai : float
            Canopy leaf area index (m2/m2).

    Returns
    -------
    flux : Flux
        Updated flux object with the following attributes:
        - rsoil : float
            Soil hydraulic resistance (MPa · s · m2/mmol H2O).
        - psi_soil : float
            Weighted soil water potential (MPa).
        - et_loss : list of float
            Fraction of total transpiration from each soil layer (-).

    """
    # Head of pressure (MPa/m) --------------------------------------------------
    # Convert mm of water to MPa
    # head =  Pressure exerted by a 1 m column of water
    # head = potential energy by unit of weight
    head = physcon.denh2o * physcon.grav * 1e-06

    # Root cross-sectional area (m2 root) ---------------------------------------
    # This represents how densely packed are the roots
    root_cross_sec_area = np.pi * rootvar.radius**2

    # Soil and root resistances for each layer ----------------------------------
    # For each layer, it computes two resistances in series
    flux.rsoil = 0.0
    psi_mpa = [0.0] * soil.nlevsoi
    evap = [0.0] * soil.nlevsoi

    for each_layer in range(soil.nlevsoi):
        # Hydraulic conductivity for each layer (mmol/m/s/MPa)
        # This represents how easily does water flow through the soil itself.
        # Computes the unsaturated hydraulic conductivity using the Clapp
        # and Hornberger (1978) power law

        # Think of the soil as a sponge with pores. When saturated (s = 1),
        # water flows easily (K = Ksat). As the soil dries (s → 0), the large
        # pores drain first, leaving only tiny tortuous pathways, and
        # conductivity drops dramatically — by orders of magnitude. The exponent
        # (2b + 3) makes this a very steep function. For loam (b = 5.39),
        # conductivity at 50% saturation is only about 0.07% of the saturated
        # value.

        s = max(min(soil.h2osoi_vol[each_layer] / soil.watsat[each_layer], 1.0), 0.01)
        hk = soil.hksat[each_layer] * s ** (2.0 * soil.bsw[each_layer] + 3.0)

        # Convert mm/s -> m2/s/MPa
        # head = potential energy by unit of weight
        hk = hk * 1e-03 / head

        # Convert m2/s/MPa -> mmol/m/s/MPa
        hk = hk * physcon.denh2o / physcon.mmh2o * 1000.0

        # Matric potential for each layer (MPa)

        # Root length density (m root per m3 soil)
        # The root length density (RLD) tells you how many metres of root exist
        # in each cubic metre of soil. From this, the code calculates the
        # average half-distance between neighbouring roots using the formula
        # for a hexagonal packing pattern

        # In a dense root zone, roots are close together
        # (small r_dist, maybe 5 mm), so water doesn't have to travel far.
        # In a sparse layer, roots are far apart (large r_dist, maybe 5 cm), a
        # nd the water's journey through the soil matrix is much longer.

        # Convert mm -> m -> MPa
        psi_mpa[each_layer] = soil.psi[each_layer] * 1e-03 * head

        # Root biomass density (g biomass / m3 soil)
        root_biomass_density = (
            rootvar.biomass * soil.rootfr[each_layer] / soil.dz[each_layer]
        )
        root_biomass_density = max(root_biomass_density, 1e-10)

        root_length_density = root_biomass_density / (
            rootvar.density * root_cross_sec_area
        )

        # Distance between roots (m)
        root_dist = np.sqrt(1.0 / (root_length_density * np.pi))

        # Soil-to-root resistance (MPa.s.m2/mmol H2O)
        # Gardner (1960) single-root model
        # The logarithmic term comes from the geometry of radial flow in a
        # cylinder (same physics as heat conduction into a pipe). The denominator
        # scales by the total root length in the layer (RLD × Δz) and
        # the soil's hydraulic conductivity K.

        # This resistance is very sensitive to soil moisture — as the soil dries,
        # K plummets, and soilr1 can increase a hundredfold.
        # This is the main "chokepoint" for water uptake in dry soil.
        soilr1 = np.log(root_dist / rootvar.radius) / (
            2.0 * np.pi * root_length_density * soil.dz[each_layer] * hk
        )

        # Root-to-stem resistance (MPa.s.m2/mmol H2O)
        # soilr1 is how hard it is for water to get to the straw
        # (travelling through the soil), and soilr2 is how hard it is to suck
        # through the straw itself (the root tissue resistance).

        soilr2 = rootvar.resist / (root_biomass_density * soil.dz[each_layer])

        # Belowground resistance (MPa.s.m2/mmol H2O)
        soilr = soilr1 + soilr2

        # Sum conductances (1/soilr) for each soil layer
        # Total belowground resistance. First sum the conductances (1/soilr)
        # for each soil layer and then convert back to a resistance after the
        # summation

        flux.rsoil += 1.0 / soilr

        # Maximum transpiration for each layer (mmol H2O/m2/s)
        # This is Ohm's law for water: flow = pressure difference / resistance.
        # The maximum possible flow from layer j is driven by the pressure
        # difference between that layer's soil water potential and the minimum
        # safe leaf water potential (the cavitation limit). If a layer is so dry
        # that its ψ_soil is below ψ_min, that layer contributes zero — it's
        # like a well that's been pumped dry.

        evap[each_layer] = (psi_mpa[each_layer] - leaf.minl_wp) / soilr
        evap[each_layer] = max(evap[each_layer], 0.0)

    # Belowground resistance: resistance = 1 / conductance ----------------------
    flux.rsoil = flux.lai / flux.rsoil

    # Weighted soil water potential (MPa) ---------------------------------------
    # The tree doesn't "see" a simple average of all soil layers. It sees a
    # transpiration-weighted average. Layers that can deliver more water
    # (wet, well-connected) get more weight. A single wet layer at 50 cm depth
    # can dominate the effective soil water potential even if all other layers
    # are bone dry.

    flux.psi_soil = 0.0
    for each_layer in range(soil.nlevsoi):
        flux.psi_soil += psi_mpa[each_layer] * evap[each_layer]

    # Total maximum transpiration
    totevap = sum(evap)

    if totevap > 0:
        flux.psi_soil /= totevap
    else:
        flux.psi_soil = leaf.minl_wp

    # Fractional transpiration uptake from soil layers --------------------------
    # This tells you what fraction of the total transpiration comes from
    # each layer. In a wet uniform profile, uptake follows the root distribution
    # (most from shallow layers where roots are densest). In a drying profile
    # with a wet deep layer, uptake shifts downward even if only a few roots are
    # there.

    flux.et_loss = []
    for each_layer in range(soil.nlevsoi):
        if totevap > 0:
            flux.et_loss.append(evap[each_layer] / totevap)
        else:
            flux.et_loss.append(1.0 / soil.nlevsoi)

    return flux

#### Example soil_resistance()